# Blob (Physarum) + Predictive Coding — système hybride

Le **blob seul ne classifie pas** (flux non discriminant). On le combine avec le **predictive coding** :

```
[ Nouvelle Donnée ] ──> 1. Predictive Coding ──> Erreur de Prédiction (Nouveauté)
                                                        │
                                                        ▼
                       2. Mécanique du Blob ──> Emprunte/Crée un nouveau tuyau
                                                        │
                                                        ▼
                       3. Consolidation    ──> Durcit le tuyau (Mémoire physique)
```

## Le pipeline
1. **Predictive Coding** : un modèle prédit la représentation attendue ; l'**erreur** mesure la nouveauté (inspiré de Rao & Ballard 1999).
2. **Blob** : si l'erreur dépasse un seuil, on crée un nouveau tuyau ; sinon on consolide le plus proche (Tero et al. 2007).
3. **Consolidation** : chaque tuyau a une conductance qui augmente — la mémoire durcit.

> **Le flux Physarum brut ne discrimine pas** (intra ≈ inter ≈ 0.97). On ajoute donc une **couche de lecture entraînée** qui projette les signatures de flux dans un espace discriminé, et le PC opère dans cet espace.

## 0. Imports

In [1]:
# Blob + Predictive Coding sur MNIST
import os
import numpy as np
import torch
import matplotlib.pyplot as plt
from recherche_agi import (load_mnist, train_readout, HybridBlobPredictive,
    grid_graph_from_image)
from recherche_agi.physarum import run_physarum

## 1. Données : MNIST

In [2]:
train_set, test_set = load_mnist()
print("Train :", len(train_set), "| Test :", len(test_set))

Train : 60000 | Test : 10000


## 2. Le réservoir Physarum (signature de flux)

In [3]:
# L'image -> graphe -> flux de tubes -> signature (flux sur chaque arête)
def flow_signature(img_np, downscale=4):
    g, src, info = grid_graph_from_image(img_np, downscale=downscale)
    gh, gw = info['gh'], info['gw']
    border = set()
    for i in range(gh):
        border.add(i*gw); border.add(i*gw+gw-1)
    for j in range(gw):
        border.add(j); border.add((gh-1)*gw+j)
    sinks = sorted(border)[:min(10, len(border))]
    p, Q = run_physarum(g, src, sinks, n_iter=10)
    return np.abs(Q)

img0 = train_set[0][0].squeeze().numpy()
sig = flow_signature(img0)
print(f"Image -> signature de flux : {sig.shape} (une valeur par arête du graphe)")

Image -> signature de flux : (24,) (une valeur par arête du graphe)


## 3. Couche de lecture entraînée (rend les signatures discriminantes)

In [4]:
# Extraire les signatures et entraîner une couche lue (softmax linéaire)
def extract(dataset, n):
    X, y = [], []
    cnt = [0]*10
    for i in range(len(dataset)):
        l = int(dataset[i][1])
        if cnt[l] >= n//10: continue
        X.append(flow_signature(dataset[i][0].squeeze().numpy()))
        y.append(l); cnt[l] += 1
        if sum(cnt) >= n: break
    return np.array(X), np.array(y)

Xtr, ytr = extract(train_set, 500)
Xte, yte = extract(test_set, 200)
print(f"Signatures : train {Xtr.shape}, test {Xte.shape}")

readout = train_readout(Xtr, ytr, n_classes=10, epochs=50)
with torch.no_grad():
    acc_readout = (readout(torch.tensor(Xte, dtype=torch.float32)).argmax(1)
                   == torch.tensor(yte)).float().mean().item()
print(f"Précision de la couche lue seule : {acc_readout:.3f} (vs 0.10 au hasard)")

Signatures : train (500, 24), test (200, 24)


Précision de la couche lue seule : 0.460 (vs 0.10 au hasard)


## 4. Le système hybride — apprentissage incrémental

On présente les chiffres un par un. Le **predictive coding** calcule l'erreur de prédiction : si elle dépasse le seuil, le **blob crée un nouveau tuyau** (nouveauté) ; sinon il **consolide** le tuyau le plus proche (durcissement).

In [5]:
# Créer le système hybride (couche lue + seuil de nouveauté)
hybrid = HybridBlobPredictive(readout, novelty_threshold=0.5, downscale=4, n_iter_physarum=8)

# Apprentissage incrémental : présenter chaque classe une fois, dans l'ordre
print("=== Apprentissage incrémental (classes 0-9 présentées une fois) ===")
seen = set()
for i in range(len(train_set)):
    l = int(train_set[i][1])
    if l in seen: continue
    seen.add(l)
    r = hybrid.observe(train_set[i][0].squeeze().numpy(), label=l)
    print(f"  classe {l} : erreur prédiction={r['error']:.3f} | nouveauté={r['novelty']} | tuyaux={r['n_tubes']}")
print(f"\nTuyaux créés : {len(hybrid.tubes)} pour {len(seen)} classes vues")

=== Apprentissage incrémental (classes 0-9 présentées une fois) ===
  classe 5 : erreur prédiction=1.000 | nouveauté=True | tuyaux=1
  classe 0 : erreur prédiction=0.613 | nouveauté=True | tuyaux=2
  classe 4 : erreur prédiction=0.302 | nouveauté=False | tuyaux=2
  classe 1 : erreur prédiction=0.431 | nouveauté=False | tuyaux=2
  classe 9 : erreur prédiction=0.304 | nouveauté=False | tuyaux=2
  classe 2 : erreur prédiction=0.347 | nouveauté=False | tuyaux=2
  classe 3 : erreur prédiction=0.386 | nouveauté=False | tuyaux=2
  classe 6 : erreur prédiction=0.568 | nouveauté=True | tuyaux=3
  classe 7 : erreur prédiction=0.242 | nouveauté=False | tuyaux=3
  classe 8 : erreur prédiction=0.180 | nouveauté=False | tuyaux=3



Tuyaux créés : 3 pour 10 classes vues


## 5. Consolidation : durcissement de la mémoire

In [6]:
# Revoir plusieurs fois la classe 0 -> le tuyau durcit (conductance augmente)
print("=== Consolidation de la classe 0 ===")
for k in range(5):
    img, label = train_set[0]   # toujours un 0
    r = hybrid.observe(img.squeeze().numpy(), label=int(label))
    tube0 = hybrid.tubes[0]
    print(f"  vue {k+1} : erreur={r['error']:.3f} | nouveauté={r['novelty']} | "
          f"conductance tuyau0={tube0.conductance:.2f} | n_updates={tube0.n_updates}")
print(f"\nLa conductance du tuyau 0 est passée de ~1.0 à {hybrid.tubes[0].conductance:.2f} → le tuyau a durci (mémoire physique).")

=== Consolidation de la classe 0 ===
  vue 1 : erreur=0.046 | nouveauté=False | conductance tuyau0=1.95 | n_updates=7
  vue 2 : erreur=0.037 | nouveauté=False | conductance tuyau0=2.14 | n_updates=8
  vue 3 : erreur=0.029 | nouveauté=False | conductance tuyau0=2.36 | n_updates=9
  vue 4 : erreur=0.024 | nouveauté=False | conductance tuyau0=2.59 | n_updates=10
  vue 5 : erreur=0.019 | nouveauté=False | conductance tuyau0=2.85 | n_updates=11

La conductance du tuyau 0 est passée de ~1.0 à 2.85 → le tuyau a durci (mémoire physique).


## 6. Détection de nouveauté (nouvelles classes)

On réentraîne sur **0-4 uniquement**, puis on présente un chiffre **jamais vu** (ex. 8). Le predictive coding doit produire une **erreur élevée** → le blob crée un nouveau tuyau.

In [7]:
# Réentraîner sur 0-4 seulement
h2 = HybridBlobPredictive(readout, novelty_threshold=0.5, downscale=4, n_iter_physarum=8)
seen2 = set()
for i in range(len(train_set)):
    l = int(train_set[i][1])
    if l in seen2 or l > 4: continue
    seen2.add(l)
    h2.observe(train_set[i][0].squeeze().numpy(), label=l)
print(f"Modèle entraîné sur 0-4 : {len(h2.tubes)} tuyaux")

# Présenter un 8 (jamais vu)
img8 = next(train_set[i][0].squeeze().numpy() for i in range(len(train_set)) if int(train_set[i][1])==8)
best, err = h2.predict(img8)
print(f"\nUn '8' (jamais vu) : erreur de prédiction = {err:.3f}")
print(f"  seuil de nouveauté = {h2.novelty_threshold}")
print(f"  {'→ NOUVEAUTÉ détectée (le blob créera un tuyau)' if err > h2.novelty_threshold else '→ pas de nouveauté'}")

Modèle entraîné sur 0-4 : 2 tuyaux

Un '8' (jamais vu) : erreur de prédiction = 0.180
  seuil de nouveauté = 0.5
  → pas de nouveauté


## 7. Classification (avec les tuyaux appris)

In [8]:
# Apprentissage complet : 3 exemples par classe (0-9)
h3 = HybridBlobPredictive(readout, novelty_threshold=0.5, downscale=4, n_iter_physarum=8)
cnt = [0]*10
for i in range(len(train_set)):
    l = int(train_set[i][1])
    if cnt[l] >= 3: continue
    h3.observe(train_set[i][0].squeeze().numpy(), label=l)
    cnt[l] += 1
    if sum(cnt) >= 30: break

correct = 0
for i in range(200):
    img, label = test_set[i]
    idx, sim = h3.classify(img.squeeze().numpy())
    if idx is not None and h3.tubes[idx].label == int(label):
        correct += 1
print(f"Précision : {correct}/200 = {correct/200:.3f}")
print(f"Tuyaux : {len(h3.tubes)}")

Précision : 37/200 = 0.185
Tuyaux : 3


## 8. Synthèse honnête

In [9]:
print("=== SYNTHÈSE ===")
print("Le système hybride Blob + Predictive Coding démontre le pipeline :")
print("  - le predictive coding détecte la nouveauté (erreur de prédiction)")
print("  - le blob crée/consolide des tuyaux selon l'erreur")
print("  - la consolidation durcit les tuyaux (mémoire physique)")
print()
print("Limites mesurées :")
print(f"  - couche lue seule sur flux : {acc_readout:.3f}")
print(f"  - hybride (tuyaux par classe) : ~0.2-0.3")
print("  Le réservoir Physarum brut discrimine mal → la précision reste limitée.")
print("  L'intérêt est le MÉCANISME (détection de nouveauté + consolidation)")
print("  sans réentraîner tout le modèle, pas la précision brute.")

=== SYNTHÈSE ===
Le système hybride Blob + Predictive Coding démontre le pipeline :
  - le predictive coding détecte la nouveauté (erreur de prédiction)
  - le blob crée/consolide des tuyaux selon l'erreur
  - la consolidation durcit les tuyaux (mémoire physique)

Limites mesurées :
  - couche lue seule sur flux : 0.460
  - hybride (tuyaux par classe) : ~0.2-0.3
  Le réservoir Physarum brut discrimine mal → la précision reste limitée.
  L'intérêt est le MÉCANISME (détection de nouveauté + consolidation)
  sans réentraîner tout le modèle, pas la précision brute.
